# 02 · Run the persona experiment

Every story is run under five conditions, and only the persona changes: no persona, mismatched nationality, matched nationality, full persona (nationality plus sex, age and field of study), and a length-matched persona that pads the matched one with a neutral sentence. Run this notebook only after the pre-registration is dated.

Any OpenAI-compatible endpoint works. For OpenAI, set `MODEL = "gpt-4o-mini"` and your `OPENAI_API_KEY`. For a local model through Ollama, set `BASE_URL = "http://localhost:11434/v1"`, `API_KEY = "ollama"` and `MODEL = "llama3.2:3b"`. `BACKEND = "mock"` gives random answers that only test the pipeline and must never be reported.

In [ ]:
import json, os, random, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from types import SimpleNamespace
import pandas as pd

## Settings

Start with `LIMIT = 20` as a pilot, inspect the outputs, then set `LIMIT = None` for the full run. The run appends to a JSONL file and skips calls already done, so you can stop and restart it.

In [ ]:
MODEL = "gpt-4o-mini"
BASE_URL = None
API_KEY = os.environ.get("OPENAI_API_KEY")
BACKEND = "api"            # "api" or "mock"
STORIES = "data/stories.csv"
RUN_NAME = MODEL.replace(":", "_").replace("/", "_") if BACKEND == "api" else "mock"
OUT = Path("results") / RUN_NAME / "raw.jsonl"   # one folder per model; mock never mixes with real runs
LIMIT = 20                 # pilot first; None = all stories
WORKERS = 8
SEED = 2026
args = SimpleNamespace(model=MODEL, backend=BACKEND, seed=SEED)

## Prompts and personas

In [ ]:
EMOTIONS = ["joy", "fear", "anger", "sadness", "disgust", "shame", "guilt"]
TASK = ("Here is a situation you experienced:\n\"{text}\"\n\n"
        "What is the main emotion you felt in this situation? Answer with exactly one word "
        "from this list: joy, fear, anger, sadness, disgust, shame, guilt.")
# Neutral, non-demographic sentences of increasing length (words), used only for padding.
NEUTRAL = ["Answer honestly.", "Please answer honestly.", "Please read this carefully.",
           "Please read the situation carefully.", "Please read the situation below carefully.",
           "Please read the situation below carefully before answering.",
           "Please read the situation below carefully and then answer honestly."]


def nationality_persona(country):
    return f"You are a person from {country}."


def full_persona(row):
    parts = [f"You are a {int(row.age)}-year-old" if pd.notna(row.age) else "You are a",
             row.sex if isinstance(row.sex, str) else "person",
             f"from {row.country}"]
    text = " ".join(parts)
    if isinstance(row.field, str):
        text += f" who studies {row.field}"
    return text + "."


def length_matched_persona(row):
    """Matched nationality persona padded with neutral sentences to the full persona's length."""
    base = nationality_persona(row.country)
    gap = len(full_persona(row).split()) - len(base.split())
    if gap <= 0:
        return base
    filler = min(NEUTRAL, key=lambda s: abs(len(s.split()) - gap))
    return base + " " + filler


def build_jobs(stories):
    countries = sorted(stories.country.unique())
    jobs = []
    for row in stories.itertuples(index=False):
        others = [c for c in countries if c != row.country]
        personas = {"none": None,
                    "matched": nationality_persona(row.country),
                    "full": full_persona(row),
                    "length_matched": length_matched_persona(row)}
        for other in others:  # with 2 countries: exactly one mismatched persona
            personas[f"mismatched::{other}"] = nationality_persona(other)
        for cond, persona in personas.items():
            condition, _, persona_country = cond.partition("::")
            if condition in ("matched", "full", "length_matched"):
                persona_country = row.country
            jobs.append({"story_id": row.story_id, "writer_id": row.writer_id,
                         "source_country": row.country, "gold": row.emotion,
                         "condition": condition, "persona_country": persona_country or "",
                         "persona": persona or "", "prompt": TASK.format(text=row.text),
                         "key": f"{row.story_id}|{cond}"})
    return jobs


## Look at the five prompts for one story

Check these before running anything: the story and the task must be identical across conditions, and the length-matched persona should have about as many words as the full persona.

In [ ]:
stories = pd.read_csv(STORIES)
if LIMIT:
    stories = stories.head(LIMIT)
jobs = build_jobs(stories)
first = stories.story_id.iloc[0]
for j in jobs:
    if j["story_id"] == first:
        print(f'{j["condition"]:<15} ({len(j["persona"].split()):>2} words) {j["persona"] or "(none)"}')
print("\n" + jobs[0]["prompt"])

## Calling the model

In [ ]:
def call_model(job, args, client):
    if args.backend == "mock":  # fake answers, only to test the pipeline end to end
        rng = random.Random(job["key"])
        return job["gold"] if rng.random() < 0.6 else rng.choice(EMOTIONS)
    messages = ([{"role": "system", "content": job["persona"]}] if job["persona"] else [])
    messages.append({"role": "user", "content": job["prompt"]})
    for attempt in range(6):
        try:
            r = client.chat.completions.create(model=args.model, messages=messages,
                                               temperature=0, max_tokens=10, seed=args.seed)
            return r.choices[0].message.content or ""
        except Exception as e:  # rate limits, timeouts
            wait = 2 ** attempt
            print(f"  retry {attempt + 1} in {wait}s: {e}")
            time.sleep(wait)
    return "__ERROR__"


## Run

In [ ]:
OUT.parent.mkdir(parents=True, exist_ok=True)
done = set()
if OUT.exists():
    done = {json.loads(l)["key"] for l in OUT.read_text(encoding="utf-8").splitlines() if l.strip()}
todo = [j for j in jobs if j["key"] not in done]
print(f"{len(jobs)} calls in total, {len(done)} already done, {len(todo)} to run.")

client = None
if BACKEND == "api":
    from openai import OpenAI
    client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

with OUT.open("a", encoding="utf-8") as f, ThreadPoolExecutor(WORKERS) as pool:
    futures = {pool.submit(call_model, j, args, client): j for j in todo}
    for i, fut in enumerate(as_completed(futures), 1):
        j = {**futures[fut], "model": RUN_NAME if BACKEND == "mock" else MODEL, "raw_output": fut.result()}
        j.pop("prompt")
        f.write(json.dumps(j) + "\n"); f.flush()
        if i % 250 == 0:
            print(f"  {i}/{len(todo)}")
print("Done:", OUT)

## Pilot check

Every output should be a single label from the list. If you see sentences, refusals or other words, fix the prompt before the full run and note the change in the pre-registration.

In [ ]:
raw = pd.read_json(OUT, lines=True)
print(raw.raw_output.str.strip().str.lower().value_counts().head(15))
raw.groupby("condition").size()